In [ ]:
from pathlib import Path
import pandas as pd

PROCESSED = Path("../data/processed")

df = pd.read_parquet(PROCESSED / "quality_events_clean.parquet")
print(df.shape)

duplicates = df[df.duplicated("event_key", keep=False)].sort_values(
    ["event_key", "batch_series", "letter_date"]
)

print("Строк в дубликатах:", len(duplicates))
duplicates.head(20)

In [ ]:
key_counts = duplicates.groupby("event_key").size().sort_values(ascending=False)
key_counts.head(10)

In [ ]:
example_key = key_counts.index[0]
df[df["event_key"] == example_key]

In [ ]:
df["event_key_v2"] = (
    df["letter_number"].fillna("") + "|" +
    df["letter_date"].dt.strftime("%Y-%m-%d").fillna("") + "|" +
    df["drug_name_clean"].fillna("") + "|" +
    df["batch_series"].fillna("") + "|" +
    df["manufacturer_clean"].fillna("") + "|" +
    df["quality_status_clean"].fillna("") + "|" +
    df["action_type_clean"].fillna("")
)

print("Строк:", len(df))
print("Уникальных event_key_v2:", df["event_key_v2"].nunique())
print("Осталось дубликатов:", len(df) - df["event_key_v2"].nunique())

In [ ]:
remaining = df[df.duplicated("event_key_v2", keep=False)]
remaining.sort_values(["event_key_v2", "letter_date"]).head(20)

In [ ]:
before = len(df)
df = df.drop_duplicates()
print(f"Удалено полных дубликатов: {before - len(df)}")

In [ ]:
quality_events = df[
    [
        "event_key_v2",
        "letter_number",
        "letter_date",
        "event_month",
        "drug_name",
        "drug_name_clean",
        "registration_number",
        "registration_date",
        "batch_series",
        "manufacturer",
        "manufacturer_clean",
        "manufacturer_country",
        "manufacturer_country_clean",
        "quality_status",
        "quality_status_clean",
        "action_type",
        "action_type_clean",
    ]
].copy()

quality_events = quality_events.rename(
    columns={
        "event_key_v2": "event_id",
        "drug_name": "drug_name_raw",
        "manufacturer": "manufacturer_raw",
        "quality_status": "quality_status_raw",
        "action_type": "action_type_raw",
        "manufacturer_country": "manufacturer_country_raw",
    }
)

quality_events.to_parquet(
    PROCESSED / "fct_quality_events.parquet",
    index=False,
)

print(quality_events.shape)
quality_events.head()

In [ ]:
monthly = (
    quality_events
    .groupby(
        [
            "manufacturer_clean",
            "drug_name_clean",
            "event_month",
        ],
        as_index=False,
    )
    .agg(
        events_count=("event_id", "nunique"),
        batches_count=("batch_series", "nunique"),
        letters_count=("letter_number", "nunique"),
        regions_affected=("manufacturer_country_clean", "nunique"),
        first_event_date=("letter_date", "min"),
        last_event_date=("letter_date", "max"),
    )
)

monthly = monthly.rename(
    columns={
        "manufacturer_clean": "manufacturer",
        "drug_name_clean": "drug_name",
    }
)

monthly["event_month"] = pd.to_datetime(monthly["event_month"] + "-01")

print(monthly.shape)
monthly.sort_values(["event_month", "events_count"], ascending=[True, False]).head(20)

In [ ]:
all_pairs = monthly[["manufacturer", "drug_name"]].drop_duplicates()

min_month = monthly["event_month"].min()
max_month = monthly["event_month"].max()

calendar = pd.date_range(min_month, max_month, freq="MS").to_frame(index=False, name="event_month")

full_grid = all_pairs.merge(calendar, how="cross")

mart = full_grid.merge(
    monthly,
    on=["manufacturer", "drug_name", "event_month"],
    how="left",
)

count_cols = ["events_count", "batches_count", "letters_count", "regions_affected"]
mart[count_cols] = mart[count_cols].fillna(0)

mart = mart.sort_values(["manufacturer", "drug_name", "event_month"])
mart.head(100)

In [ ]:
group_cols = ["manufacturer", "drug_name"]

mart["events_lag_1m"] = mart.groupby(group_cols)["events_count"].shift(1)
mart["events_lag_3m"] = mart.groupby(group_cols)["events_count"].shift(3)
mart["events_lag_6m"] = mart.groupby(group_cols)["events_count"].shift(6)

mart["events_rolling_3m"] = (
    mart.groupby(group_cols)["events_count"]
    .transform(lambda s: s.rolling(3, min_periods=1).sum())
)

mart["events_rolling_6m"] = (
    mart.groupby(group_cols)["events_count"]
    .transform(lambda s: s.rolling(6, min_periods=1).sum())
)

mart["events_rolling_12m"] = (
    mart.groupby(group_cols)["events_count"]
    .transform(lambda s: s.rolling(12, min_periods=1).sum())
)

mart["events_rolling_3m_mean"] = (
    mart.groupby(group_cols)["events_count"]
    .transform(lambda s: s.rolling(3, min_periods=1).mean())
)

mart["events_rolling_12m_mean"] = (
    mart.groupby(group_cols)["events_count"]
    .transform(lambda s: s.rolling(12, min_periods=1).mean())
)

In [ ]:
mart["risk_flag_spike"] = (
    (mart["events_count"] >= 2)
    & (mart["events_count"] >= 3 * mart["events_rolling_12m_mean"])
).astype(int)

mart["risk_flag_repeated"] = (
    mart["events_rolling_3m"] >= 2
).astype(int)

mart["risk_flag_active_month"] = (mart["events_count"] > 0).astype(int)

mart["risk_score_v0"] = (
    2 * mart["risk_flag_spike"]
    + 1 * mart["risk_flag_repeated"]
    + 1 * mart["risk_flag_active_month"]
)

In [ ]:
top_risk = (
    mart[mart["event_month"] == mart["event_month"].max()]
    .sort_values(["risk_score_v0", "events_count"], ascending=[False, False])
    .head(20)
)

top_risk[
    [
        "event_month",
        "manufacturer",
        "drug_name",
        "events_count",
        "events_rolling_3m",
        "events_rolling_12m_mean",
        "risk_score_v0",
    ]
]

In [ ]:
mart.to_parquet(
    PROCESSED / "mart_manufacturer_drug_month.parquet",
    index=False,
)

In [ ]:
assert mart.duplicated(["manufacturer", "drug_name", "event_month"]).sum() == 0
assert mart["events_count"].min() >= 0
assert mart["events_lag_1m"].isna().sum() > 0  # у первых месяцев нет прошлого
print("Витрина валидна")
print(mart.shape)

In [ ]:
mart.shape
mart["event_month"].min(), mart["event_month"].max()
mart["events_count"].describe()
mart["risk_flag_spike"].value_counts()
top_risk.head(10)